# S5b — the last four measurements: under-bust, front length, high hip, knee (Colab, CPU is enough)

specs/s5b-geometry-rough/spec.md (approved 25 Nov 2026). Trains nothing. Front length comes from ANSUR's seated
front length (`differs`, D-033); under-bust, high hip and knee come from the outline and are `rough`, with no
± error until your tape data exists (D-034). The two bust-point rows stay not available (D-025).


## 1. Drive and data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/anthro"
DRIVE_ARCHIVE = "/content/drive/MyDrive/datasets/bodym.tar"
DATA_DIR = "/content/bodym"
os.makedirs(DRIVE, exist_ok=True)

if os.path.exists(DATA_DIR):
    print("Data already on this runtime.")
elif os.path.exists(DRIVE_ARCHIVE):
    !tar -xf "{DRIVE_ARCHIVE}" -C /content
else:
    !pip -q install awscli
    !aws s3 sync --no-sign-request s3://amazon-bodym/ {DATA_DIR} --only-show-errors
    os.makedirs(os.path.dirname(DRIVE_ARCHIVE), exist_ok=True)
    !tar -cf "{DRIVE_ARCHIVE}" -C /content bodym

## 2. Get the code

In [ ]:
BRANCH = "s5b-geometry-rough"   # set to "main" once merged
REPO = "https://github.com/Anthropometry/Anthropometry_Accademic_Model.git"
!rm -rf /content/repo && git clone -q --branch {BRANCH} {REPO} /content/repo
!pip -q install -e /content/repo
!git -C /content/repo log -1 --format="commit %h  %s"
GEO, CACHE = f"{DRIVE}/geo", f"{DRIVE}/cache"
RUN = f"{DRIVE}/runs/r4"
ANSUR = f"{DRIVE}/ansur2"
FORMULAS = f"{ANSUR}/ansur_formulas.json"

# A package installed in a running Colab session is only importable after a restart;
# pointing Python at the code directly avoids that.
import sys
sys.path.insert(0, "/content/repo/src")


## 3. Which ANSUR columns could match? (seconds)
**Paste back:** the table.


In [ ]:
!anthro ansur-columns --ansur {ANSUR}


## 4. Refit the ANSUR formulas (adds front length and ANSUR's knee height), then the mix decision
Same as notebook 05, cells 4 and 5. **Paste back:** the first lines of cell 4 (up to the first table) and the
front_length rows.


In [ ]:
!anthro ansur-fit --ansur {ANSUR} --report {RUN}/eval_report.json --out {FORMULAS} --bodym {DATA_DIR}


In [ ]:
!anthro ansur-mix --data {DATA_DIR} --ckpt {RUN}/best.pt --formulas {FORMULAS} --cache {CACHE} --geo-cache {GEO}


## 5. Rough outline estimates on the 202 validation people (a few minutes)
Spread and how often each code could be measured, by sex and BMI band. A sanity check, not a score.
**Paste back:** everything this cell prints.


In [ ]:
!anthro geometry-rough --data {DATA_DIR} --ckpt {RUN}/best.pt --formulas {FORMULAS}


## 6. One Test-B woman and one man: the full set

In [ ]:
from anthro.data import load_split
tb = load_split(DATA_DIR, "testB")
woman = tb[tb.gender == "female"].iloc[0]
man = tb[tb.gender == "male"].iloc[0]
for p in (woman, man):
    print(p.photo_id, p.gender, round(p.height_cm, 1), "cm", round(p.weight_kg, 1), "kg")


def mask(p, side):
    return f"{DATA_DIR}/testB/{'mask_left' if side else 'mask'}/{p.photo_id}.png"


In [ ]:
WF, WS = mask(woman, 0), mask(woman, 1)
WH, WW = f"{woman.height_cm:.1f}", f"{woman.weight_kg:.1f}"
!anthro predict-silhouettes --front {WF} --side {WS} --height {WH} --weight {WW} --sex female --age-range 25-34 --ckpt {RUN}/best.pt --ansur {FORMULAS}


In [ ]:
MF, MS = mask(man, 0), mask(man, 1)
MH, MW = f"{man.height_cm:.1f}", f"{man.weight_kg:.1f}"
!anthro predict-silhouettes --front {MF} --side {MS} --height {MH} --weight {MW} --sex male --age-range 25-34 --ckpt {RUN}/best.pt --ansur {FORMULAS}


## 7. Send the formula file
Attach the new `ansur_formulas.json` (in `MyDrive/anthro/ansur2/`): it now holds front length and the knee
height, and replaces the packaged one.


## Before committing this notebook
**Edit → Clear all outputs**, then save. CI refuses notebooks with saved outputs (POLICY 6).
